<a href="https://colab.research.google.com/github/seankh06/Flyrank-ML-Engineer-Internship-Starter/blob/main/work/scripts/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seankh06/Flyrank-ML-Engineer-Internship-Starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import duckdb
from google.colab import userdata

# ambil token asli dari Secrets
hf_token = userdata.get('HF_TOKEN')

# connect ke duckdb
con = duckdb.connect()

# kasih token asli (bukan placeholder)
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# test query kecil
rel = "hf://datasets/FlyRank/internship-warehouse"
result = con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')").fetchall()
print(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[(78835655,)]


## 1. Unit of analysis + time window

One row represents one content page, on one specific day. This is confirmed in Section 3: content_hash_id values repeat exactly 31 times within month=2026-03, matching the number of days in March. For the actual analysis, I aggregate this daily data into monthly totals per page, so each page ends up represented once per month, similar to how the starter dataset's _90d columns work.

## 2. Fields: feature / label / context / excluded

I'll use two tables: dim_content, to get word_count and other page metadata, and fact_content_daily_performance, to get daily session counts that I'll aggregate into a monthly total. I'll join them on content_hash_id.

## 3. Verify it with queries (grain, counts, missing values, windows)


In [ ]:
query1 = f"""
SELECT content_hash_id, COUNT(*) as row_count
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY content_hash_id
HAVING COUNT(*) > 1
LIMIT 10
"""
result1 = con.sql(query1).fetchall()
print("Duplicate content_hash_id in month=2026-03:", len(result1))
print(result1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate content_hash_id in month=2026-03: 10
[('content_b7e512995f79d5a6', 31), ('content_05597932fe4da067', 31), ('content_905aa32a0230694e', 31), ('content_05434271b257bb68', 31), ('content_d056587ff7faca0c', 31), ('content_bfd1e41c2af250c8', 31), ('content_2662845f598544ef', 31), ('content_22610b0934f8825e', 31), ('content_712c365258cee05c', 31), ('content_476c37c366920c1b', 31)]


In [ ]:
query2 = f"""
SELECT COUNT(*) as total_rows, MIN(report_date) as earliest_date, MAX(report_date) as latest_date
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
result2 = con.sql(query2).fetchall()
print(result2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[(9841378, datetime.date(2026, 3, 1), datetime.date(2026, 3, 31))]


In [ ]:
query3 = f"""
SELECT COUNT(*) as available_rows
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE ga4_data_available IS TRUE
"""
result3 = con.sql(query3).fetchall()
print(result3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[(413966,)]


The first query confirms the grain: content_hash_id values repeat exactly 31 times in month=2026-03, matching March's 31 days, meaning the raw table is one row per page per day. This daily data gets aggregated into monthly totals per page for the actual analysis.

The second query shows the slice contains 9,841,378 rows, spanning the full month from March 1 to March 31, 2026, confirming the date window is complete with no gaps at the edges.

The third query checks availability using ga4_data_available IS TRUE, and only 413,966 rows pass that filter, about 4.2% of the total. This means the vast majority of rows in this month don't have GA4 session data available, likely because many clients' GA4 tracking wasn't active yet or wasn't linked for that period. This is an important limitation to keep in mind since the lane depends on session data specifically.

## Five features + the leakage trap

Building a small feature frame from month=2026-03, then deliberately adding one label-derived
column to see the score jump, before removing it and keeping the honest number.

In [ ]:
# 1. Agregat data harian jadi bulanan, per halaman
query_features = f"""
SELECT
    content_hash_id,
    SUM(ga4_sessions) as sessions_month,
    SUM(gsc_impressions) as impressions_month,
    AVG(gsc_avg_position) as avg_position_month,
    COUNT(*) as days_with_data
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY content_hash_id
"""
df_features = con.sql(query_features).df()

# 2. Ambil word_count dan content_type dari dim_content
query_content = f"""
SELECT content_hash_id, word_count, content_type
FROM read_parquet('{rel}/dim_content.parquet')
"""
df_content = con.sql(query_content).df()

# 3. Gabungin keduanya
df_merged = df_features.merge(df_content, on="content_hash_id", how="inner")

# 4. Bikin target dari median sessions
median_sessions = df_merged["sessions_month"].median()
df_merged["high_engagement"] = (df_merged["sessions_month"] >= median_sessions).astype(int)

print(df_merged.shape)
df_merged[["word_count", "avg_position_month", "content_type", "days_with_data",
           "impressions_month", "high_engagement"]].head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331437, 8)


,word_count,avg_position_month,content_type,days_with_data,impressions_month,high_engagement
0,2123,7.209549,keyword article,31,6523.0,1
1,<NA>,2.987198,keyword article,31,453.0,1
2,2546,6.724039,keyword article,31,5630.0,1
3,2330,7.244844,keyword article,31,4944.0,1
4,<NA>,14.432540,keyword article,31,42.0,1
5,<NA>,4.209227,keyword article,31,429.0,1
6,<NA>,9.445635,keyword article,31,223.0,1
7,<NA>,6.014516,keyword article,31,96.0,1
8,<NA>,9.155335,keyword article,31,314.0,1
9,2556,5.258331,keyword article,31,7709.0,1


Five features, each with why it's knowable at the decision moment:

1. **word_count** — known at publish time, before any traffic happens.
2. **avg_position_month** — average search ranking position, largely settled from indexing and
   optimization, not caused by future clicks.
3. **content_type** — metadata set when the page is created.
4. **days_with_data** — how many days the page had tracked activity in the month, known once
   the month closes, used only for the training window.
5. **impressions_month** — how many times the page appeared in search results this month. This
   one is borderline since it occurs in the same window as sessions, so it's treated as
   descriptive context rather than a fully safe predictive feature.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

df_leak = df_merged.dropna(subset=["word_count"]).copy()
df_leak["sessions_leak"] = df_leak["sessions_month"]  # sengaja bocor: turunan langsung dari target

X_leaky = df_leak[["word_count", "avg_position_month", "days_with_data", "sessions_leak"]]
y = df_leak["high_engagement"]

X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.3, random_state=42)
tree_leaky = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_leaky.fit(X_train, y_train)

leaky_score = tree_leaky.score(X_test, y_test)
print(f"Accuracy WITH leakage (sessions_leak included): {leaky_score:.3f}")

Accuracy WITH leakage (sessions_leak included): 1.000


In [ ]:
# hapus kolom bocor, latih ulang model
X_honest = df_leak[["word_count", "avg_position_month", "days_with_data"]]

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
tree_honest = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_honest.fit(X_train, y_train)

honest_score = tree_honest.score(X_test, y_test)
print(f"Accuracy WITHOUT leakage: {honest_score:.3f}")

Accuracy WITHOUT leakage: 0.869


## 4. Data limits

This data can't tell us about pages before their client's GSC or GA4 tracking started, since those rows either don't exist or have missing metrics. It also can't separate genuine engagement from seasonal or one-time traffic spikes within the month, since we're only looking
at monthly totals, not the pattern across days. Only about 4.2% of rows had ga4_data_available marked TRUE this month, so the session-based analysis only reflects a small, possibly non-representative slice of the full content inventory.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.